# RIN — golden-микромир: ретрив против известного ответа

Зачем этот ноутбук, отдельно от `embryonic_graph.ipynb`: там — большой реальный эмбрион, без эталона;
здесь — маленький ручно-написанный граф с ЗАРАНЕЕ известными правильными ответами на запросы.

**Это фикстура, не вторая механика** (`F_stage1.tex`: «synthetic tests — маленький граф с ground truth», последний шаг pipeline v0).
Граф статичен, руками написан, ничего живого его не мутирует (ни fusion, ни write-path, ни сон) — те же
операторы (`query_vectors`/`nearest_nodes_by_embedding`/`diffuse`/`pack_context`), что и на эмбрионе, просто на другом графе.

**Цель:** увидеть семантический вентиль (`rel(B|q)`) в `diffuse()` на графе, где есть известный правильный ответ —
чтобы можно было измерить, а не судить на глаз.

In [1]:
import sys
from pathlib import Path

def find_repo_root(marker="requirements.txt"):
    # Notebook now lives in src/experiments/, whose cwd in Jupyter is
    # usually the notebook's own directory, not the repo root — walk up
    # until a known repo-root marker is found, instead of assuming cwd.
    path = Path.cwd()
    for candidate in [path, *path.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"could not find repo root (looked for {marker})")

REPO_ROOT = find_repo_root()
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

In [2]:
%load_ext autoreload
%autoreload 2

## Статус

Граф ещё не написан — состав сценариев обсуждаем отдельно (см. переписку выше). Намечено:

1. `Azores / multi-word` — longest-match уже готов и проверен отдельно (см. `embryonic/query.py`), здесь будет его сценарий на ручном графе.
2. `Distractor vs answer` — главный: доказательство для `rel(B|q)`.
3. `Мульти-якорный запрос` — оба топика должны представиться.
4. `Нерелевантное слово` — регресс-тест на пустой якорь.

ATOMIC в этом ноутбуке не участвует вообще (отключён по плану — шумелий слой, к корректности ретрива не относится).

## Загрузка Numberbatch

Нужен для реальных векторов узлов микромира — чтобы тест семантического вентиля был честным (definitions.tex: один энкодер для query и для node.embedding), а не подогнанными векторами.

In [3]:
from pathlib import Path
from embryonic.numberbatch import Numberbatch

NUMBERBATCH_FILE = REPO_ROOT / "source" / "numberbatch-en-19.08.txt.gz"
numberbatch = Numberbatch()
numberbatch.load(str(NUMBERBATCH_FILE))
print(f"{len(numberbatch.terms):,} терминов")

516,782 терминов


## Граф

Две темы, покрывающие все 4 сценария:

- **coffee/caffeine/beverage** — главный сценарий для семантического вентиля. `caffeine` — конкретный, по-делу правильный ответ на «почему кофе бодрит». `beverage` — дистрактор-хаб: общий, связан с многим и (через `coffee` И `hot drink`) получает усиление с двух сторон — точно так же, как `lisbon` получил 146.67 вместо 100 на реальном графе.
- **north_dakota/south_dakota/north** — сценарий для longest-match: без него «north dakota» развалилось бы на несвязанные `north` + `dakota`.

In [4]:
from memory.graph import MemoryGraph
from memory.node import Node
from memory.edge import Edge

NODE_SPECS = [
    # (content, numberbatch_uri, node_kind, confidence)
    ("coffee", "/c/en/coffee", "concept", 0.7),
    ("caffeine", "/c/en/caffeine", "concept", 0.7),
    ("beverage", "/c/en/beverage", "concept", 0.7),
    ("hot drink", "/c/en/hot_drink", "concept", 0.7),
    ("tea", "/c/en/tea", "concept", 0.7),
    ("juice", "/c/en/juice", "concept", 0.7),
    ("soda", "/c/en/soda", "concept", 0.7),
    ("north dakota", "/c/en/north_dakota", "concept", 0.7),
    ("south dakota", "/c/en/south_dakota", "concept", 0.7),
    ("north", "/c/en/north", "concept", 0.7),
]

EDGE_SPECS = [
    # (src_content, dst_content, type, weight)
    ("coffee", "caffeine", "HasProperty", 1.0),
    ("coffee", "beverage", "IsA", 1.0),
    ("coffee", "hot drink", "RelatedTo", 1.0),
    ("hot drink", "beverage", "IsA", 1.0),
    ("beverage", "tea", "RelatedTo", 1.0),
    ("beverage", "juice", "RelatedTo", 1.0),
    ("beverage", "soda", "RelatedTo", 1.0),
    ("north dakota", "south dakota", "RelatedTo", 1.0),
]

G = MemoryGraph()
content_to_id = {}
for i, (content, uri, kind, conf) in enumerate(NODE_SPECS):
    vector = numberbatch.vector_for(uri)
    assert vector is not None, f"no vector for {uri}"
    G.add_node(Node(id=i, content=content, embedding=vector, node_kind=kind,
                     confidence=conf, provenance=[uri]))
    content_to_id[content] = i

for src, dst, etype, weight in EDGE_SPECS:
    G.add_edge(Edge(src=content_to_id[src], dst=content_to_id[dst], type=etype, weight=weight))
    G.add_edge(Edge(src=content_to_id[dst], dst=content_to_id[src], type=etype, weight=weight))

print(f"{len(G.nodes)} узлов, все с настоящими Numberbatch-векторами")

10 узлов, все с настоящими Numberbatch-векторами


## Сценарий 1: Azores/multi-word — longest-match

Ожидаем: якорь — `north dakota` как единая единица, НЕ `north` + `dakota` по отдельности.

In [5]:
from embryonic.query import longest_match_terms, query_vectors
from memory.retrieval import nearest_nodes_by_embedding

QUERY_1 = "tell me about north dakota"
terms = longest_match_terms(QUERY_1, numberbatch)
print("совпавшие термины:", terms)

vecs = query_vectors(QUERY_1, numberbatch)
anchors_1 = set()
for v in vecs:
    matches = nearest_nodes_by_embedding(G, v, top_k=3)
    anchors_1.update(matches)
    print("  ->", [G.nodes[m].content for m in matches])

print("\nЯкоря:", [G.nodes[a].content for a in anchors_1])
print("ОЖИДАЕМ: \u201cnorth dakota\u201d в якорях, \u201cnorth\u201d сам по себе — нет (или сильно ниже по рангу)")

совпавшие термины: ['/c/en/north_dakota']
  -> ['north dakota', 'south dakota']

Якоря: ['south dakota', 'north dakota']
ОЖИДАЕМ: “north dakota” в якорях, “north” сам по себе — нет (или сильно ниже по рангу)


## Сценарий 2: distractor vs answer — главный тест

Запрос не упоминает «caffeine» напрямую — чтобы проверялась диффузия (шаг 2), а не якорение (шаг 1).

**Ожидаемая проблема без вентиля:** `beverage` получает заряд и от `coffee`, и от `hot drink` — два источника, суммируются, может обогнать `caffeine` (один источник), хотя `beverage` не отвечает на вопрос «почему энергия» по сути.

In [6]:
from memory.diffuse import diffuse
from memory.packer import pack_context, to_text

QUERY_2 = "why coffee gives energy"
vecs_2 = query_vectors(QUERY_2, numberbatch)
print("query terms matched:", longest_match_terms(QUERY_2, numberbatch))

anchors_2 = set()
for v in vecs_2:
    matches = nearest_nodes_by_embedding(G, v, top_k=3)
    anchors_2.update(matches)
    print("  ->", [G.nodes[m].content for m in matches])

print("\nЯкоря:", [G.nodes[a].content for a in anchors_2])

source_amplitudes_2 = {a: 100.0 for a in anchors_2}
lit_2 = diffuse(G, source_amplitudes_2, decay=0.8, min_propagate_fraction=0.01, max_steps=20)

print("\nРейтинг по пик-амплитуде (БЕЗ семантического вентиля):")
for nid, amp in sorted(lit_2.items(), key=lambda kv: -kv[1]):
    marker = ""
    if G.nodes[nid].content == "caffeine":
        marker = " <-- ожидаемый ответ"
    elif G.nodes[nid].content == "beverage":
        marker = " <-- дистрактор"
    print(f"  {G.nodes[nid].content!r:15s} {amp:7.2f}{marker}")

query terms matched: ['/c/en/coffee', '/c/en/gives', '/c/en/energy']
  -> ['coffee', 'hot drink', 'tea']
  -> []
  -> []

Якоря: ['coffee', 'hot drink', 'tea']

Рейтинг по пик-амплитуде (БЕЗ семантического вентиля):
  'beverage'       146.67 <-- дистрактор
  'coffee'         100.00
  'hot drink'      100.00
  'tea'            100.00
  'caffeine'        26.67 <-- ожидаемый ответ
  'juice'           23.47
  'soda'            23.47


## Сценарий 3: мульти-якорный запрос

Обе темы (coffee и north dakota) должны представиться в финальной выдаче.

In [7]:
QUERY_3 = "coffee and north dakota"
vecs_3 = query_vectors(QUERY_3, numberbatch)
anchors_3 = set()
for v in vecs_3:
    anchors_3.update(nearest_nodes_by_embedding(G, v, top_k=3))

source_amplitudes_3 = {a: 100.0 for a in anchors_3}
lit_3 = diffuse(G, source_amplitudes_3, decay=0.8, min_propagate_fraction=0.01, max_steps=20)
packed_3 = pack_context(G, lit_3, token_budget=200)
print(to_text(packed_3))

[prior] beverage: IsA coffee, IsA hot drink, RelatedTo tea, RelatedTo juice, RelatedTo soda
[prior] coffee: HasProperty caffeine, IsA beverage, RelatedTo hot drink
[prior] hot drink: RelatedTo coffee, IsA beverage
[prior] tea: RelatedTo beverage
[prior] north dakota: RelatedTo south dakota
[prior] south dakota: RelatedTo north dakota
[prior] caffeine: HasProperty coffee
[prior] juice: RelatedTo beverage
[prior] soda: RelatedTo beverage


## Сценарий 4: нерелевантное слово (регресс)

In [8]:
QUERY_4 = "what is the difference between coffee and tea"
terms_4 = longest_match_terms(QUERY_4, numberbatch)
print("совпавшие термины:", terms_4)
print("ОЖИДАЕМ: ни одного стоп-слова (what/is/the/between/and) среди них")

совпавшие термины: ['/c/en/difference', '/c/en/between', '/c/en/coffee', '/c/en/tea']
ОЖИДАЕМ: ни одного стоп-слова (what/is/the/between/and) среди них
